# 01 · Data overview

First look at the raw Kaggle dataset (`arshkon/linkedin-job-postings`) **before** any cleaning:
which files exist, how big they are, what columns and types they have, how much is missing,
and how the experience-level field (our future seniority label) is distributed.

Run `python -m src.data.download` first.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
assert (RAW / "postings.csv").exists(), "Run `python -m src.data.download` first"


## 1. Files, sizes, row and column counts

In [2]:
def inventory(raw_dir: Path) -> pd.DataFrame:
    rows = []
    for f in sorted(raw_dir.rglob("*.csv")):
        df = pd.read_csv(f, low_memory=False)
        rows.append(
            {
                "file": f.relative_to(raw_dir).as_posix(),
                "size_mb": round(f.stat().st_size / 1e6, 1),
                "rows": len(df),
                "columns": df.shape[1],
            }
        )
    return pd.DataFrame(rows).sort_values("size_mb", ascending=False, ignore_index=True)


inventory(RAW)

,file,size_mb,rows,columns
0,postings.csv,516.8,123849,31
1,companies/companies.csv,23.2,24473,10
2,companies/company_specialities.csv,4.4,169387,2
3,jobs/job_skills.csv,3.5,213768,2
4,jobs/job_industries.csv,2.5,164808,2
5,jobs/salaries.csv,2.3,40785,8
6,jobs/benefits.csv,1.9,67943,3
7,companies/employee_counts.csv,1.0,35787,4
8,companies/company_industries.csv,0.8,24375,2
9,mappings/industries.csv,0.0,422,2


## 2. The main table: `postings.csv`

Loaded once. Descriptions contain line breaks inside quoted fields, so we use pandas' default C parser.

In [3]:
postings = pd.read_csv(RAW / "postings.csv", low_memory=False)
print(f"{postings.shape[0]:,} rows × {postings.shape[1]} columns")
print(f"In-memory size: {postings.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"Unique job_id: {postings['job_id'].nunique():,}")

123,849 rows × 31 columns
In-memory size: 533 MB
Unique job_id: 123,849


## 3. Columns, data types, and missing percentage

In [4]:
overview = pd.DataFrame(
    {
        "dtype": postings.dtypes.astype(str),
        "non_null": postings.notna().sum(),
        "missing_pct": (postings.isna().mean() * 100).round(1),
        "n_unique": postings.nunique(),
        "example": postings.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    }
)
overview["example"] = overview["example"].astype(str).str.slice(0, 60)
overview.sort_values("missing_pct")

,dtype,non_null,missing_pct,n_unique,example
job_id,int64,123849,0.0,123849,921716
title,str,123849,0.0,72521,Marketing Coordinator
description,str,123842,0.0,107827,Job descriptionA leading real estate firm in N...
location,str,123849,0.0,8526,"Princeton, NJ"
original_listed_time,float64,123849,0.0,65036,1713397508000.0
job_posting_url,str,123849,0.0,123849,https://www.linkedin.com/jobs/view/921716/?trk...
formatted_work_type,str,123849,0.0,7,Full-time
application_type,str,123849,0.0,4,ComplexOnsiteApply
sponsored,int64,123849,0.0,1,0
listed_time,float64,123849,0.0,53231,1713397508000.0


## 4. Experience level (the future seniority label)

In [5]:
exp = postings["formatted_experience_level"].value_counts(dropna=False)
pd.DataFrame({"count": exp, "pct": (exp / len(postings) * 100).round(1)})

,count,pct
formatted_experience_level,,
Mid-Senior level,41489,33.5
Entry level,36708,29.6
NaN,29409,23.7
Associate,9826,7.9
Director,3746,3.0
Internship,1449,1.2
Executive,1222,1.0


## 5. Other fields that shape the cleaning plan

In [6]:
for col in ["pay_period", "formatted_work_type", "remote_allowed", "currency", "compensation_type"]:
    print(f"--- {col}")
    print(postings[col].value_counts(dropna=False).to_string(), end="\n\n")

--- pay_period
pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9

--- formatted_work_type
formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487

--- remote_allowed
remote_allowed
NaN    108603
1.0     15246

--- currency
currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2

--- compensation_type
compensation_type
NaN            87776
BASE_SALARY    36073



In [7]:
# listed_time is a Unix timestamp in milliseconds
listed = pd.to_datetime(postings["listed_time"], unit="ms")
print("Listed from", listed.min(), "to", listed.max())
listed.dt.to_period("M").value_counts().sort_index()

Listed from 2024-03-24 21:50:14 to 2024-04-20 00:26:56


listed_time
2024-03         1
2024-04    123848
Freq: M, Name: count, dtype: int64

In [8]:
# Salary columns: which postings have any salary info, and how does normalized_salary compare?
has_salary = postings[["min_salary", "med_salary", "max_salary"]].notna().any(axis=1)
print(f"Postings with any salary: {has_salary.mean():.1%}")
postings.loc[has_salary, ["min_salary", "med_salary", "max_salary", "normalized_salary"]].describe().round(0)

Postings with any salary: 29.1%


,min_salary,med_salary,max_salary,normalized_salary
count,29793.0,6280.0,29793.0,36073.0
mean,64911.0,22016.0,91939.0,205327.0
std,495974.0,52256.0,701110.0,5097627.0
min,1.0,0.0,1.0,0.0
25%,37.0,19.0,48.0,52000.0
50%,60000.0,26.0,80000.0,81500.0
75%,100000.0,2510.0,140000.0,125000.0
max,85000000.0,750000.0,120000000.0,535600000.0


In [9]:
# Location format (to plan parsing in Phase 2)
postings["location"].value_counts().head(15)

location
United States                      8125
New York, NY                       2756
Chicago, IL                        1834
Houston, TX                        1762
Dallas, TX                         1383
Atlanta, GA                        1363
Boston, MA                         1176
Austin, TX                         1083
Charlotte, NC                      1075
Phoenix, AZ                        1059
Washington, DC                      985
Los Angeles, CA                     972
San Francisco, CA                   884
New York City Metropolitan Area     837
Seattle, WA                         818
Name: count, dtype: int64

## 6. Join tables

In [10]:
job_skills = pd.read_csv(RAW / "jobs" / "job_skills.csv")
skill_map = pd.read_csv(RAW / "mappings" / "skills.csv")
job_industries = pd.read_csv(RAW / "jobs" / "job_industries.csv")
industry_map = pd.read_csv(RAW / "mappings" / "industries.csv")

print(f"skills.csv: {len(skill_map)} skill categories")
print(f"job_skills: {len(job_skills):,} rows, {job_skills['job_id'].nunique():,} jobs, "
      f"{job_skills.groupby('job_id').size().mean():.2f} categories per job")
print(f"job_industries: {len(job_industries):,} rows, {job_industries['job_id'].nunique():,} jobs")
print(f"industries.csv: {len(industry_map)} industries")

(job_skills.merge(skill_map, on="skill_abr")["skill_name"].value_counts().head(15))

skills.csv: 35 skill categories


job_skills: 213,768 rows, 126,807 jobs, 1.69 categories per job
job_industries: 164,808 rows, 127,125 jobs
industries.csv: 422 industries


skill_name
Information Technology    26137
Sales                     22475
Management                20861
Manufacturing             18185
Health Care Provider      17369
Business Development      14290
Engineering               13009
Other                     12608
Finance                    8540
Marketing                  5525
Accounting/Auditing        5461
Administrative             4860
Customer Service           4292
Project Management         3997
Analyst                    3858
Name: count, dtype: int64

In [11]:
(job_industries.merge(industry_map, on="industry_id")["industry_name"].value_counts().head(15))

industry_name
Hospitals and Health Care             18326
Retail                                11033
IT Services and IT Consulting         10396
Staffing and Recruiting                9005
Financial Services                     8535
Software Development                   5091
Manufacturing                          3689
Construction                           3445
Banking                                2923
Insurance                              2673
Pharmaceutical Manufacturing           2469
Hospitality                            2455
Telecommunications                     2433
Real Estate                            2326
Industrial Machinery Manufacturing     2143
Name: count, dtype: int64

## Observations (input to Phase 2)

- **123,849 postings × 31 columns**, one row per `job_id`. The dataset covers *all* industries (hospitals and retail are the top two), so a tech-title filter is essential.
- **Time window is narrow:** 99% of postings were listed in **April 2024** (even by `original_listed_time`, only ~1.8k are from March). A "trend over time" analysis is limited to about 4 weeks.
- **Experience level is missing for 23.7%** of postings. Mapped to our 3 classes: Entry ≈ 38k, Mid ≈ 51k, **Senior ≈ 5k (≈5% of labeled rows)**. That's a strong class imbalance.
- LinkedIn's **"Mid-Senior level"** is the largest bucket (41k) and mixes mid and senior roles. That makes the Mid/Senior boundary noisy and should be discussed in Phase 2.
- **Salary is present for only 29%** of postings, in mixed units (`YEARLY`, `HOURLY`, `MONTHLY`, `WEEKLY`, `BIWEEKLY`). The provided `normalized_salary` has extreme outliers (max ≈ $536M), so our own normalization + IQR filter is needed.
- `remote_allowed` is only ever `1` or missing, so missing likely means "not stated". Hybrid is not a separate field and will have to come from text.
- `location` is mostly "City, ST". About 8k rows are just "United States", with no city/state.
- `job_skills` holds **LinkedIn's 35 broad categories** (IT, Sales, Engineering, ...), not tech skills. Our own skill extractor (Phase 4) is required.
- Almost all salaries are USD (only ~15 in other currencies). These can be dropped.
